<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 14 · 持续接入项目文档

项目目录每天都在变化。我们用真实 OpenDAL 文件后端扫描两份文档，修改其中一份，再加入不可解码的文件，观察 checkpoint 如何决定能否继续。

需要 Python 3.12+ 和 notebooks 依赖组；无需模型。使用现有 evaluation Connector，展示不可变 Source 快照、持久化回执与 checkpoint，不宣称它实现完整的逻辑 Source 历史或自动 Memory 消费。

路线：首次接入 → 原样重跑 → 修改 → 故障 → 恢复 → Server 重启后继续。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show, table

from powercontext.http import CreateScopeRequest

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))
if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("14", features=())
client = lab.client
assert client is not None
scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 14", summary="本次教学实验的独立材料", idempotency_key=f"{lab.run_id}:main"
    )
)
scope_id = scope.scope_id

## 给 Worker 一个明确的目录和 Scope

文件读取发生在 Worker；Server 接收已物化的材料。这里选本机目录，换成远端存储时凭证仍应只留在 Worker 配置中。

In [ ]:
from powercontext_connector_opendal import TEXT_FILE_SNAPSHOT_SOURCE_DEFINITION, OpenDALTextFileConnector

from powercontext.client import RemoteConnectorWorker
from powercontext.sources import ConnectorBinding, SourceDefinitionRegistry

materials = lab.directory / "project"
(materials / "docs").mkdir(parents=True)
(materials / "docs" / "amount.md").write_text("amount: 金额以整数分存储。", encoding="utf-8")
(materials / "docs" / "errors.md").write_text("错误提示包含原始行号。", encoding="utf-8")
connector = OpenDALTextFileConnector.from_service(
    "fs", source_namespace=lab.run_id, root="docs", storage_options={"root": str(materials)}
)
binding = ConnectorBinding(
    scope_id=scope_id,
    binding_id=f"{lab.run_id}-docs",
    connector_name=connector.name,
    connector_version=connector.version,
)
registry = SourceDefinitionRegistry((TEXT_FILE_SNAPSHOT_SOURCE_DEFINITION,))
worker = RemoteConnectorWorker(client=client, registry=registry)
first = await worker.run(connector, binding)
assert len(first.items) == 2 and all(item.status == "accepted" for item in first.items)
assert first.committed_checkpoint is not None
show(first)

## 内容未变化时，不重复提交

先原样运行，再只修改 amount 文档。新的文件摘要对应一个新的不可变 Source；旧快照仍可以准确读取。

In [ ]:
same = await worker.run(connector, binding)
assert len(same.items) == 0
(materials / "docs" / "amount.md").write_text("amount: 金额以整数分存储；拒绝超过两位小数。", encoding="utf-8")
changed = await worker.run(connector, binding)
assert len(changed.items) == 1 and changed.items[0].status == "accepted"
assert changed.items[0].source_ref not in [item.source_ref for item in first.items]
table([
    {"运行": "首次", "提交项": len(first.items)},
    {"运行": "未修改", "提交项": len(same.items)},
    {"运行": "修改一份", "提交项": len(changed.items)},
])

## 一个坏文件，为什么不能推进整个 checkpoint？

写入不可解码的字节，触发真实 UTF-8 解码拒绝。拒绝是本轮执行结果的一部分；不能因为别的文件成功，就把这一轮标记为全部完成。

In [ ]:
broken = materials / "docs" / "broken.txt"
broken.write_bytes(bytes([255, 254, 253]))
failed = await worker.run(connector, binding)
assert any(item.status == "rejected" for item in failed.items)
assert failed.committed_checkpoint == failed.previous_checkpoint
show({"本轮状态": failed.status, "checkpoint 未前移": True})
broken.write_text("修复后的 UTF-8 文档。", encoding="utf-8")
recovered = await worker.run(connector, binding)
assert all(item.status == "accepted" for item in recovered.items) and len(recovered.items) == 1
assert recovered.committed_checkpoint != failed.previous_checkpoint

## 重启服务后，进度仍在

Worker 从服务端读取已持久化的 checkpoint。接入成功本身不创建 Memory；我们同时检查这个容易混淆的边界。

In [ ]:
from powercontext.http import ListMemoryEntriesRequest

await lab.restart()
client = lab.client
resumed = await RemoteConnectorWorker(client=client, registry=registry).run(connector, binding)
assert len(resumed.items) == 0
memory = await client.list_memory_entries(ListMemoryEntriesRequest(scope_id=scope_id))
assert memory.entries == []
print("重启后无重复提交；Source 接入没有直接创建 Memory。")

## 练习与验收

删除一个文件后重跑，观察 checkpoint 的路径集合。删除文件不会删除已接收的 Source。可继续尝试修改两个文件，预期只有这两份产生新快照。

接下来阅读 [15_background_learning.ipynb](15_background_learning.ipynb)。

最后关闭服务。实验文件保留在本次 `.powercontext/` 目录，便于复查。

In [ ]:
await lab.close()
print("本篇 Server 已关闭。")